# **MODEL 1: Multimodal Integration - Late Fusion (EfficientNetB1 + Metadata MLP)**

**Objective:**
Inclusion of **clinical metadata**
- **Age**
- **Sex**
- **Atomic location** of the lesion
<br>

**Methodology:**
| Variable | Strategy | NA Treatment |
|----------|-----------|--------------------|
| Age | Normalization (min-max) | Imputation by median |
| Sex | One-hot encoding | Category "unknown" |
| Location | One-hot encoding (15 sítios) | Category "unknown" |

### Imports

In [23]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize

# Add project root to sys.path so local utils are importable
sys.path.insert(0, os.path.abspath(".."))
from utils.utils_preproc import create_dl_splits, format_center_crop_tf
from utils.utils_model import get_callbacks, plot_history


### **Data** Configuration

In [24]:
if os.getcwd().endswith('models'):
    os.chdir('..')

In [25]:
BASE_PATH  = "./data"
"""
AUG_DIR    = os.path.join(BASE_PATH, "HAM10000_augmented")
META_PATH  = os.path.join(BASE_PATH, "HAM10000_metadata")
LABEL_PATH = os.path.join(BASE_PATH, "..", "label2idx.json")   # repo root
"""

'\nAUG_DIR    = os.path.join(BASE_PATH, "HAM10000_augmented")\nMETA_PATH  = os.path.join(BASE_PATH, "HAM10000_metadata")\nLABEL_PATH = os.path.join(BASE_PATH, "..", "label2idx.json")   # repo root\n'

In [31]:
# Image directories
AUG_IMG_DIR = os.path.join(BASE_PATH, "HAM10000_augmented")       # augmented images (train)
VAL_IMG_DIRS = [
    os.path.join(BASE_PATH, "HAM10000_images_part_1"),
    os.path.join(BASE_PATH, "HAM10000_images_part_2"),
    os.path.join(BASE_PATH, "HAM10000ISIC2018_Task3_Test_Images"), #add part 3 !!!!!!!!
   
]

# Metadata CSVs — splits already done upstream
AUG_META_PATH  = os.path.join(BASE_PATH, "augmented_metadata.csv")  # train 
VAL_META_PATH  = os.path.join(BASE_PATH, "val_split.csv")
TEST_META_PATH = os.path.join(BASE_PATH, "test_split.csv")

In [27]:
LABEL_PATH = os.path.join(BASE_PATH, "..", "label2idx.json")

In [28]:
N_CLASSES  = 7
BATCH_SIZE = 32
IMG_SIZE   = 224

### **Data** Loading & Mapping

In [29]:
# Label encoding (shared across all splits)
with open(LABEL_PATH) as fh:
    label2idx = json.load(fh)
idx2label = {v: k for k, v in label2idx.items()}

In [45]:
# 3.2  TRAIN  (augmented data)
aug_df = pd.read_csv(AUG_META_PATH) #augmented_metadata

aug_img_map = {
    os.path.splitext(f)[0]: os.path.join(AUG_IMG_DIR, f) #HAM10000_augmented
    for f in os.listdir(AUG_IMG_DIR)
    if f.lower().endswith(".jpg")
}
aug_df["image_path"] = aug_df["image_id"].map(aug_img_map)
#aug_df = aug_df.dropna(subset=["image_path"]).reset_index(drop=True) # PODEMOS APAGAR
print(f"Train (augmented) images linked: {len(aug_df)}")

if "dx_encoded" not in aug_df.columns:
    aug_df["dx_encoded"] = aug_df["dx"].map(label2idx)


Train (augmented) images linked: 12621


In [46]:
aug_df.head()

,image_id,dataset,lesion_id,image_path,dx,dx_encoded,age,sex,localization
0,ISIC_0024313,rosendahl,HAM_0002869,NaN,mel,4,50.0,female,back
1,ISIC_0024315,vidir_modern,HAM_0007538,NaN,mel,4,55.0,male,trunk
2,ISIC_0024318,vidir_modern,HAM_0002450,NaN,df,3,65.0,female,lower extremity
3,ISIC_0024323,rosendahl,HAM_0002493,NaN,mel,4,50.0,male,lower extremity
4,ISIC_0024324,vidir_modern,HAM_0000351,NaN,bkl,2,85.0,male,back


In [38]:
# VAL & TEST  (original images)
orig_img_map = {}
for d in VAL_IMG_DIRS:
    if os.path.exists(d):
        for f in os.listdir(d):
            if f.lower().endswith(".jpg"):
                orig_img_map[os.path.splitext(f)[0]] = os.path.join(d, f)

val_df  = pd.read_csv(VAL_META_PATH)
test_df = pd.read_csv(TEST_META_PATH)

for split_df, name in [(val_df, "Val"), (test_df, "Test")]:
    split_df["image_path"] = split_df["image_id"].map(orig_img_map)
    split_df.dropna(subset=["image_path"], inplace=True)
    split_df.reset_index(drop=True, inplace=True)
    if "dx_encoded" not in split_df.columns:
        split_df["dx_encoded"] = split_df["dx"].map(label2idx)
    print(f"{name} images linked: {len(split_df)}")

Val images linked: 983
Test images linked: 979


In [34]:
#  MLP metadata features
"""
Expected columns already in aug_df (produced upstream):
    age_norm  → float, MinMax-scaled age
    sex_ohe   → one-hot encoded columns for sex
    loc_ohe   → one-hot encoded columns for localization
"""

sex_ohe_cols = [c for c in aug_df.columns if c.startswith("sex_")]
loc_ohe_cols = [c for c in aug_df.columns if c.startswith("localization_")]

META_COLS = ["age_norm"] + sex_ohe_cols + loc_ohe_cols
N_META    = len(META_COLS)
print(f"MLP input dimension: {N_META} features → {META_COLS}")

MLP input dimension: 1 features → ['age_norm']


## Metadata preprocessing

Clean and encode the clinical features (Age, Sex, Localization) into a fixed-length vector before the split.

In [39]:
# Identify feature columns from aug_df
sex_ohe_cols = sorted([c for c in aug_df.columns if c.startswith("sex_")])
loc_ohe_cols = sorted([c for c in aug_df.columns if c.startswith("localization_")])
META_COLS    = ["age_norm"] + sex_ohe_cols + loc_ohe_cols
META_DIM     = len(META_COLS)
print(f"MLP input dim: {META_DIM}  →  {META_COLS}")

MLP input dim: 1  →  ['age_norm']


In [43]:
# Fill missing age_norm in augmented rows with train mean
age_mean = aug_df["age"].mean()
aug_df["age_norm"] = aug_df["age"].fillna(age_mean)

nan


In [ ]:
# ── Correlação metadados × diagnóstico ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Metadata by Diagnostoc", fontsize=14)

# Age by diagnosis
classes_order = sorted(df["dx"].unique())
age_by_dx = [df[df["dx"] == c]["age_clean"].values for c in classes_order]
axes[0].boxplot(age_by_dx, labels=classes_order)
axes[0].set_title("Faixa Etária por Diagnóstico")
axes[0].set_xlabel("Diagnostic")
axes[0].set_ylabel("Age")
axes[0].tick_params(axis='x', rotation=30)

# Sex distribution per diagnosis
sex_dx = df.groupby(["dx", "sex"]).size().unstack(fill_value=0)
sex_dx.plot(kind="bar", ax=axes[1], stacked=True, 
            color=["#4E79A7", "#F28E2B", "#E15759"])
axes[1].set_title("Distribuição de Sexo por Diagnóstico")
axes[1].set_xlabel("Diagnóstico")
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title="Sexo")

plt.tight_layout()
plt.show()

## 4. Multimodal Data Pipeline

In [ ]:
BATCH_SIZE = 32

def load_multimodal_item(img_path, meta_vec, label):
    # Image Branch
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32)
    img = format_center_crop_tf(img) 
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    
    # Returning a dictionary matching input names in build_late_fusion_model
    return {"image_input": img, "meta_input": meta_vec}, label

def make_ds(df_split, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        df_split["image_path"].values,
        np.stack(df_split["meta_vector"].values),
        df_split["dx_encoded"].values.astype(np.int32)
    ))
    if shuffle: ds = ds.shuffle(2000)
    return ds.map(load_multimodal_item).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_df, shuffle=True)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

## 5. Architecture Late Fusion: EfficientNetB1 + Metadata MLP 

This builds the two branches and concatenates them into the final classification head.

In [ ]:
def build_late_fusion_model(meta_dim, n_classes=7):
    # Image Branch
    img_input = keras.Input(shape=(224, 224, 3), name="image_input")
    backbone = keras.applications.EfficientNetB1(include_top=False, weights="imagenet", input_tensor=img_input)
    backbone.trainable = False  # Frozen for Phase 1

    x = keras.layers.GlobalAveragePooling2D()(backbone.output)
    x = keras.layers.Dense(256, activation="relu")(x)
    img_features = keras.layers.Dropout(0.4)(x)

    # Metadata Branch
    meta_input = keras.Input(shape=(meta_dim,), name="meta_input")
    m = keras.layers.Dense(64, activation="relu")(meta_input)
    m = keras.layers.BatchNormalization()(m)
    meta_features = keras.layers.Dense(32, activation="relu")(m)

    # Late Fusion
    combined = keras.layers.Concatenate()([img_features, meta_features])
    z = keras.layers.Dense(128, activation="relu")(combined)
    out = keras.layers.Dense(n_classes, name="logits")(z)

    return keras.Model(inputs=[img_input, meta_input], outputs=out)

model = build_late_fusion_model(META_DIM)
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [ ]:
# PAST VERSION !!

""" N_CLASSES = len(label2idx)


def build_late_fusion_model(meta_dim, n_classes=7, freeze_backbone=True):
    -
    Constrói um modelo de Late Fusion com dois ramos:
      - Ramo de Imagem: EfficientNetB1 (pré-treinada no ImageNet)
      - Ramo de Metadados: MLP pequena
    
    Os dois ramos são concatenados antes da camada de classificação final.
    
    Parameters
    ----------
    meta_dim     : int   — Dimensão do vetor de metadados
    n_classes    : int   — Número de classes de saída
    freeze_backbone : bool — Se True, congela o backbone na Fase 1
    
    Returns
    -------
    keras.Model com duas entradas: (img_input, meta_input)
    -
    # ── Ramo de Imagem ──────────────────────────────────────────────────────
    img_input = keras.Input(shape=(224, 224, 3), name="image_input")

    backbone = keras.applications.EfficientNetB1(
        include_top=False,
        weights="imagenet",
        input_tensor=img_input,
    )
    backbone.trainable = not freeze_backbone

    x = backbone.output
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = keras.layers.Dense(256, activation="relu", name="img_dense")(x)
    x = keras.layers.Dropout(0.4, name="img_dropout")(x)
    img_features = x  # shape: (batch, 256)

    # ── Ramo de Metadados (MLP) ─────────────────────────────────────────────
    meta_input = keras.Input(shape=(meta_dim,), name="meta_input")

    m = keras.layers.Dense(64, activation="relu", name="meta_dense1")(meta_input)
    m = keras.layers.BatchNormalization(name="meta_bn1")(m)
    m = keras.layers.Dropout(0.3, name="meta_dropout1")(m)
    m = keras.layers.Dense(32, activation="relu", name="meta_dense2")(m)
    meta_features = m  # shape: (batch, 32)

    # ── Integração Late Fusion ──────────────────────────────────────────────
    combined = keras.layers.Concatenate(name="late_fusion")([img_features, meta_features])

    out = keras.layers.Dense(128, activation="relu", name="fusion_dense")(combined)
    out = keras.layers.Dropout(0.3, name="fusion_dropout")(out)
    out = keras.layers.Dense(n_classes, name="logits")(out)  # raw logits

    model = keras.Model(
        inputs=[img_input, meta_input],
        outputs=out,
        name="late_fusion_model",
    )
    return model


model = build_late_fusion_model(META_DIM, N_CLASSES, freeze_backbone=True)
model.summary(show_trainable=True)
"""

## 6. Phase 1: Training the Fusion Head (Backbone Frozen)

In [ ]:
# Callbacks from your utils
callbacks = get_callbacks(model_name="late_fusion_phase1")

# Phase 1 Training
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

plot_history(history_phase1)

## 7. Phase 2: Fine-Tuning (Unfrozen)

In [ ]:
# Unfreeze the backbone
for layer in model.layers:
    if "efficientnet" in layer.name:
        layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(1e-5), # Very low learning rate
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

callbacks_ft = get_callbacks(model_name="late_fusion_finetuned")

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_ft
)

## 8. Final Evaluation & Confusion Matrix

In [ ]:
# Predict
y_pred_logits = model.predict(test_ds)
y_pred = np.argmax(y_pred_logits, axis=1)
y_true = test_df["dx_encoded"].values

# Classification Report
print(classification_report(y_true, y_pred, target_names=list(label2idx.keys())))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=label2idx.keys(), yticklabels=label2idx.keys())
plt.title("Confusion Matrix - Multimodal Late Fusion")
plt.show()

## 9. Ablation Study: What is the clinical data worth?
To see if the metadata actually contributes to the model, we run the test set again but "blind" the model by passing zero vectors as metadata.

In [ ]:
# Create a "Blind" metadata set (all zeros)
test_meta_zeros = np.zeros_like(np.stack(test_df["meta_vector"].values))

blind_ds = tf.data.Dataset.from_tensor_slices((
    {"image_input": test_df["image_path"].values, "meta_input": test_meta_zeros},
    test_df["dx_encoded"].values
)).map(lambda x, y: ({"image_input": load_multimodal_item(x["image_input"], x["meta_input"], y)[0]["image_input"], 
                      "meta_input": x["meta_input"]}, y)).batch(BATCH_SIZE)

# Evaluate Blind vs Multimodal
loss_m, acc_m = model.evaluate(test_ds)
loss_b, acc_b = model.evaluate(blind_ds)

print(f"Accuracy with Metadata: {acc_m:.4f}")
print(f"Accuracy without Metadata (Blind): {acc_b:.4f}")
print(f"Clinical Lift: {acc_m - acc_b:.4f}")